# Acciones por pantalla: fine-tuning de FunctionGemma en Colab

Usa los mismos scripts de `projects/mobile/ml/acciones/` (no duplica su código). No usa Drive: todo corre en el disco temporal de Colab.

**Antes de empezar**
1. Runtime > Change runtime type > GPU (A100 si hay; Google reporta ~60 min con su dataset grande).
2. Ten a mano los 6 archivos de `acciones/` (`acciones_pantallas.json`, `ejemplos.jsonl`, `ejemplos_prueba.jsonl` y los 3 scripts `.py`): se suben desde tu PC.
3. Acepta la licencia de `google/functiongemma-270m-it` en Hugging Face y ten un token a mano.

In [ ]:
# Versiones con las que Google probó su notebook de Mobile Actions
!pip install -q transformers==4.57.1 trl==0.25.1 datasets==4.4.1 accelerate

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Sube los 6 archivos de la carpeta acciones/ (disco temporal de Colab, no Drive)
import os
from google.colab import files

RUTA = '/content/acciones'
os.makedirs(RUTA, exist_ok=True)
%cd {RUTA}
files.upload()
!ls

## 1. Dataset (catálogo + ejemplos → entrenamiento/validación)

In [ ]:
# 350 por acción (~3.150 ejemplos): con datos de plantilla, más no aporta y alarga el entrenamiento
!python generar_datos.py --max-por-accion 350

## 2. Entrenar

Valores por defecto del notebook de Mobile Actions de Google (2 épocas, lr 1e-5, `cosine`), con lote 2 y acumulación 4 para que quepa en la T4. El script mide el largo de los ejemplos, imprime un ejemplo renderizado y avisa si alguno se cortaría. **Fíjate en la pérdida (`loss`) de los primeros pasos: si sale 0.0, algo está mal, para el entrenamiento.** Solo se guarda el último checkpoint.

In [ ]:
!python entrenar_acciones.py

## 3. Evaluar

Primero con las frases escritas a mano (`prueba.jsonl`), que el modelo nunca vio y son la medida real; después con la validación generada, que suele dar más alto.

In [ ]:
!python evaluar_acciones.py --datos datos/prueba.jsonl
!python evaluar_acciones.py --datos datos/validacion.jsonl --errores 0

## 4. Exportar a LiteRT (`.litertlm`) para el teléfono

**Sin verificar:** esta celda sigue lo que describe el notebook de Mobile Actions de Google (según un resumen, no el código literal). Los nombres de la API de `ai-edge-torch` pueden haber cambiado; si falla, compara con el [notebook original](https://github.com/google-gemini/gemma-cookbook/tree/main/FunctionGemma).

In [ ]:
!pip install -q ai-edge-torch-nightly ai-edge-litert-nightly

from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter

modelo = gemma3.build_model_270m(f'{RUTA}/salida/modelo')
converter.convert_to_litert(
    modelo,
    output_path=f'{RUTA}/salida',
    output_name_prefix='acciones',
    prefill_seq_len=256,
    kv_cache_max_len=1024,
    quantize='dynamic_int8',
    output_format='litertlm',
)
!ls -lh {RUTA}/salida

## 5. Bajar el resultado a tu PC

El disco de Colab se borra al terminar la sesión: descarga esto antes de cerrarla. El `.litertlm` cuantizado pesa cerca de 300 MB; el modelo fusionado (`salida/modelo`), ~1 GB.

In [ ]:
import glob

for f in glob.glob(f'{RUTA}/salida/*.litertlm'):
    files.download(f)

# Opcional: el modelo completo (para reexportar o evaluar después)
# !cd {RUTA}/salida && zip -r modelo.zip modelo
# files.download(f'{RUTA}/salida/modelo.zip')